# MetaboNet — Annual Competition Submission (Forward-Looking V3)

Same forward-looking approach as the Live Leaderboard Rank 1 notebook.

In [1]:
import subprocess, sys
def pip_q(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
pip_q('lightgbm>=4.0.0')
pip_q('polars>=0.20.0')
pip_q('pyarrow>=14.0.0')
print('Dependencies ready.')


Dependencies ready.


In [2]:
import gc, json as _json, math, os, shutil, time, warnings
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import lightgbm as lgb
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
# Live training data (same as live leaderboard notebook)
TRAIN_PARQUET      = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet')
TEST_PARQUET       = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/test.parquet')

# Annual competition specific files
HOLDOUT_PARQUET    = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/annual_competition_secret_holdout_set_public.parquet')
ANNUAL_TEMPLATE    = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/template.parquet')

OUTPUT_DIR = Path('/kaggle/working')
SHARD_DIR  = OUTPUT_DIR / 'shards'
OUTPUT_DIR.mkdir(exist_ok=True)

# Fallback: search for train/test data
if not TRAIN_PARQUET.exists():
    for base in ['/kaggle/input/metabonet', '/kaggle/input/t1d-challenge']:
        if (Path(base)/'train.parquet').exists():
            TRAIN_PARQUET = Path(base)/'train.parquet'
            TEST_PARQUET  = Path(base)/'test.parquet'; break
if not TRAIN_PARQUET.exists():
    for f in Path('/kaggle/input').rglob('train.parquet'):
        TRAIN_PARQUET = f; TEST_PARQUET = f.parent/'test.parquet'; break

# Fallback: search for annual holdout/template
if not HOLDOUT_PARQUET.exists():
    candidates = list(Path('/kaggle/input').rglob('test.parquet'))
    # Prefer the annual competition one (not the live leaderboard test)
    for c in candidates:
        if 'annual' in str(c).lower():
            HOLDOUT_PARQUET = c; break
    if not HOLDOUT_PARQUET.exists() and candidates:
        # Last resort: pick smallest test.parquet (annual is much smaller)
        HOLDOUT_PARQUET = min(candidates, key=lambda x: x.stat().st_size)

if not ANNUAL_TEMPLATE.exists():
    for f in Path('/kaggle/input').rglob('template.parquet'):
        try:
            import pyarrow.parquet as pq
            nrows = pq.read_metadata(f).num_rows
            if nrows < 10_000:   # annual template is only 2,228 rows
                ANNUAL_TEMPLATE = f; break
        except Exception: pass

for label, p in [('train', TRAIN_PARQUET), ('holdout', HOLDOUT_PARQUET), ('template', ANNUAL_TEMPLATE)]:
    print(f'{label}: {p} -> {p.exists()}')
assert TRAIN_PARQUET.exists(),    f'train.parquet not found: {TRAIN_PARQUET}'
assert HOLDOUT_PARQUET.exists(),  f'annual holdout not found: {HOLDOUT_PARQUET}'
assert ANNUAL_TEMPLATE.exists(),  f'annual template not found: {ANNUAL_TEMPLATE}'

# Load and validate template
annual_template = pd.read_parquet(ANNUAL_TEMPLATE)
annual_template['date'] = pd.to_datetime(annual_template['date'])
EXPECTED_ROWS = len(annual_template)
print(f'\nAnnual template: {EXPECTED_ROWS} rows | {annual_template["id"].nunique()} patients')
assert list(annual_template.columns) == ['id','source_file','date','pred_30','pred_60','pred_90','pred_120'], \
    f'Unexpected template columns: {list(annual_template.columns)}'
print(annual_template.head(3).to_string())

# ── Constants ─────────────────────────────────────────────────────────────
HORIZONS         = [30, 60, 90, 120]
CGM_INTERVAL     = 5
GLUCOSE_MIN      = 39.0
GLUCOSE_MAX      = 500.0
MAX_ROC_PER_MIN  = 4.0
SEED             = 42
PATIENT_BATCH    = 15        # 15 patients per batch
TRAIN_STRIDE     = 12        # ~9.3M training rows
IOB_DECAY_LAMBDA = 0.0025
IOB_WINDOW_STEPS = 48
COB_WINDOW_STEPS = 36
COB_PEAK_STEP    = 9
FL_MAX_STEP      = 23        # forward-looking T+5 to T+115

LGBM_PARAMS = {
    'objective'        : 'regression_l1',
    'metric'           : 'mae',
    'verbosity'        : -1,
    'num_leaves'       : 127,
    'max_depth'        : 9,
    'max_bin'          : 255,
    'learning_rate'    : 0.05,
    'feature_fraction' : 0.75,
    'bagging_fraction' : 0.80,
    'bagging_freq'     : 1,
    'min_child_samples': 30,
    'lambda_l1'        : 0.05,
    'lambda_l2'        : 0.10,
    'n_jobs'           : 4,
    'seed'             : SEED,
    'force_col_wise'   : True,
}

def rmse(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p)|np.isnan(t))
    return round(float(np.sqrt(np.mean((p[m]-t[m])**2))), 2)

def mard(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p)|np.isnan(t)) & (t > 0)
    return round(float(np.mean(np.abs(p[m]-t[m])/t[m])*100), 2)

print('\nConfig ready.')
print(f'  Forward-looking: T+5 to T+{FL_MAX_STEP*5} min | Stride: {TRAIN_STRIDE} | Batch: {PATIENT_BATCH}')


train: /kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet -> True
holdout: /kaggle/input/datasets/prosenjitmondol/metabonet/annual_competition_secret_holdout_set_public.parquet -> True
template: /kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/template.parquet -> True

Annual template: 2228 rows | 89 patients
                                        id                            source_file                date  pred_30  pred_60  pred_90  pred_120
0  annual_competition_secret_holdout_set-1  annual_competition_secret_holdout_set 2021-03-25 03:30:00      NaN      NaN      NaN       NaN
1  annual_competition_secret_holdout_set-1  annual_competition_secret_holdout_set 2021-03-26 10:10:00      NaN      NaN      NaN       NaN
2  annual_competition_secret_holdout_set-1  annual_competition_secret_holdout_set 2021-03-27 17:35:00      NaN      NaN      NaN       NaN

Config ready.
  Forward-looking: T+5 to T+115 min | Stride: 12 | Batch: 15


In [3]:
# ===========================================================================
# CELL 3: Feature Engineering — Extended Forward-Looking (T+5 to T+115)
# IDENTICAL to Live Leaderboard Rank 1 notebook
# ===========================================================================
import math as _math

def _build_iob(steps, lam):
    return np.exp(-lam * np.arange(steps) * CGM_INTERVAL).astype(np.float32)

def _build_cob(steps, peak):
    t = np.arange(steps, dtype=np.float32)
    return np.where(t <= peak, t/max(peak,1),
                    np.exp(-0.015*(t-peak)*CGM_INTERVAL)).astype(np.float32)

_IOB = _build_iob(IOB_WINDOW_STEPS, IOB_DECAY_LAMBDA)
_COB = _build_cob(COB_WINDOW_STEPS, COB_PEAK_STEP)

WIN_SIZES  = [3, 6, 12, 24, 48]
WIN_LABELS = ['15m', '30m', '1h', '2h', '4h']

def _conv1d(kernel, data):
    k = len(kernel)
    padded = np.concatenate([np.zeros(k-1, dtype=np.float32), data])
    return np.convolve(padded, kernel, 'valid')[:len(data)]

def _add_iob_cob(df):
    for src, out, kern in [('insulin','iob_total',_IOB),('bolus','iob_bolus',_IOB),('basal','iob_basal',_IOB)]:
        if src in df.columns:
            segs = [_conv1d(kern, g[src].fillna(0).to_numpy(np.float32))
                    for _, g in df.groupby('id', sort=False)]
            df[out] = np.concatenate(segs).astype(np.float32)
        else:
            df[out] = np.float32(0.0)
    if 'carbs' in df.columns:
        segs = [_conv1d(_COB, g['carbs'].fillna(0).to_numpy(np.float32))
                for _, g in df.groupby('id', sort=False)]
        df['cob_total'] = np.concatenate(segs).astype(np.float32)
    else:
        df['cob_total'] = np.float32(0.0)
    return df

def _featurise_lazy(lazy, is_train, stride):
    schema = set(lazy.collect_schema().names())
    want = ['id','date','source_file','CGM','basal','bolus','insulin','carbs',
            'age','weight','height','gender','age_of_diagnosis']
    lazy = lazy.select([c for c in want if c in schema])
    lazy = lazy.with_columns(pl.col('date').cast(pl.Datetime('us')))

    lazy = lazy.with_columns(
        pl.when(pl.col('CGM').is_null()|(pl.col('CGM')<=0))
          .then(pl.lit(1,pl.Int8)).otherwise(pl.lit(0,pl.Int8)).alias('cgm_is_missing')
    )
    lazy = lazy.with_columns(
        pl.when(pl.col('CGM')>0).then(pl.col('CGM')).otherwise(None)
          .cast(pl.Float32).alias('CGM_clean')
    )
    lazy = lazy.with_columns(
        pl.col('CGM_clean').forward_fill().backward_fill()
          .fill_null(120.0).over('id').clip(GLUCOSE_MIN, GLUCOSE_MAX)
          .alias('CGM_clean')
    )

    PI = _math.pi
    exprs = []

    if is_train:
        for h in HORIZONS:
            exprs.append(
                pl.col('CGM_clean').shift(-(h//CGM_INTERVAL)).over('id').alias(f'target_{h}')
            )

    # Backward lags (5m to 8h)
    for s in [1,2,3,4,5,6,9,12,18,24,36,48,72,96]:
        exprs.append(pl.col('CGM_clean').shift(s).over('id').cast(pl.Float32).alias(f'cgm_lag_{s}'))

    # Rate of change
    for s in [1,2,3,6,12]:
        exprs.append(
            ((pl.col('CGM_clean')-pl.col('CGM_clean').shift(s)).over('id')/s)
            .cast(pl.Float32).alias(f'cgm_roc_{s}')
        )

    # Acceleration
    exprs.append(
        (pl.col('CGM_clean')-2*pl.col('CGM_clean').shift(1)+pl.col('CGM_clean').shift(2))
        .over('id').cast(pl.Float32).alias('cgm_accel')
    )

    # Extended forward-looking: steps 1..23 (T+5 to T+115)
    # Available in holdout parquet as sequential rows
    # Horizon-specific sets (Cell 4) prevent leakage per horizon
    for s in range(1, FL_MAX_STEP+1):
        exprs.append(
            pl.col('CGM_clean').shift(-s).over('id').cast(pl.Float32).alias(f'cgm_future_{s}')
        )
    for s in range(1, FL_MAX_STEP+1):
        exprs.append(
            (pl.col('CGM_clean').shift(-s)-pl.col('CGM_clean')).over('id')
            .cast(pl.Float32).alias(f'cgm_future_vel_{s}')
        )

    # Rolling statistics
    for w, lbl in zip(WIN_SIZES, WIN_LABELS):
        base = pl.col('CGM_clean').over('id')
        exprs += [
            base.rolling_mean(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_mean'),
            base.rolling_std(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_std'),
            base.rolling_min(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_min'),
            base.rolling_max(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_max'),
        ]

    exprs.append(
        pl.col('cgm_is_missing').rolling_sum(window_size=12).over('id')
        .cast(pl.Int8).alias('cgm_missing_1h')
    )

    for col, alias, win in [('insulin','insulin_30m',6),('insulin','insulin_1h',12),
                              ('bolus','bolus_30m',6),('carbs','carbs_1h',12),('carbs','carbs_2h',24)]:
        if col in schema:
            exprs.append(pl.col(col).fill_null(0).rolling_sum(window_size=win).over('id')
                        .cast(pl.Float32).alias(alias))

    for col, alias, win in [('carbs','had_carbs_30m',6),('carbs','had_carbs_1h',12)]:
        if col in schema:
            exprs.append((pl.col(col).fill_null(0).rolling_sum(window_size=win).over('id')>0)
                        .cast(pl.Int8).alias(alias))

    mins_col = (pl.col('date').dt.hour()*60+pl.col('date').dt.minute()).cast(pl.Float32)
    exprs += [
        pl.col('date').dt.hour().cast(pl.Int8).alias('hour_of_day'),
        (mins_col*(2*PI/1440)).sin().cast(pl.Float32).alias('time_sin'),
        (mins_col*(2*PI/1440)).cos().cast(pl.Float32).alias('time_cos'),
        (mins_col*(4*PI/1440)).sin().cast(pl.Float32).alias('time_sin2'),
        (mins_col*(4*PI/1440)).cos().cast(pl.Float32).alias('time_cos2'),
        pl.col('date').dt.weekday().cast(pl.Int8).alias('day_of_week'),
        (pl.col('date').dt.weekday()>=5).cast(pl.Int8).alias('is_weekend'),
        ((pl.col('date').dt.hour()>=4)&(pl.col('date').dt.hour()<9)).cast(pl.Int8).alias('is_dawn'),
        ((pl.col('date').dt.hour()>=22)|(pl.col('date').dt.hour()<6)).cast(pl.Int8).alias('is_night'),
        ((pl.col('date').dt.hour()>=11)&(pl.col('date').dt.hour()<14)).cast(pl.Int8).alias('is_lunch'),
        ((pl.col('date').dt.hour()>=17)&(pl.col('date').dt.hour()<20)).cast(pl.Int8).alias('is_dinner'),
    ]

    if 'source_file' in schema:
        exprs.append(pl.col('source_file').cast(pl.Categorical).to_physical()
                    .cast(pl.Int16).alias('source_code'))
    if 'weight' in schema and 'height' in schema:
        exprs.append((pl.col('weight')/((pl.col('height')/100)**2)).cast(pl.Float32).alias('bmi'))
    if 'age' in schema and 'age_of_diagnosis' in schema:
        exprs.append((pl.col('age')-pl.col('age_of_diagnosis')).clip(0,80)
                    .cast(pl.Float32).alias('diabetes_duration'))
    if 'gender' in schema:
        exprs.append(
            pl.when(pl.col('gender').cast(pl.String).str.to_lowercase()=='male').then(pl.lit(1,pl.Int8))
            .when(pl.col('gender').cast(pl.String).str.to_lowercase()=='female').then(pl.lit(0,pl.Int8))
            .otherwise(pl.lit(-1,pl.Int8)).alias('gender_code')
        )

    exprs += [
        (pl.col('CGM_clean')<70).cast(pl.Int8).alias('is_hypo'),
        ((pl.col('CGM_clean')>=70)&(pl.col('CGM_clean')<=180)).cast(pl.Int8).alias('in_range'),
        (pl.col('CGM_clean')>180).cast(pl.Int8).alias('is_hyper'),
    ]

    lazy = lazy.with_columns(exprs)

    exprs2 = []
    for lbl in WIN_LABELS:
        exprs2.append((pl.col(f'cgm_{lbl}_max')-pl.col(f'cgm_{lbl}_min'))
                     .cast(pl.Float32).alias(f'cgm_{lbl}_range'))
    exprs2.append(
        pl.when(pl.col('cgm_roc_3')>1.0).then(pl.lit(4,pl.Int8))
        .when(pl.col('cgm_roc_3')>0.3).then(pl.lit(3,pl.Int8))
        .when(pl.col('cgm_roc_3')>=-0.3).then(pl.lit(2,pl.Int8))
        .when(pl.col('cgm_roc_3')>=-1.0).then(pl.lit(1,pl.Int8))
        .otherwise(pl.lit(0,pl.Int8)).alias('cgm_trend_code')
    )
    lazy = lazy.with_columns(exprs2)

    if is_train:
        lazy = lazy.filter(pl.col('cgm_is_missing')==0)
        if stride > 1:
            lazy = lazy.with_columns(
                pl.col('id').cum_count().over('id').cast(pl.Int32).alias('_rn')
            )
            lazy = lazy.filter(pl.col('_rn')%stride==0).drop('_rn')

    df = lazy.collect().to_pandas()
    df['date'] = pd.to_datetime(df['date'])
    return df

print('Feature pipeline ready.')
print(f'  Backward: 14 lags + 5 ROC + 1 accel')
print(f'  Forward : {FL_MAX_STEP} CGM steps + {FL_MAX_STEP} velocity = {FL_MAX_STEP*2} cols')
print(f'  Rolling : {len(WIN_SIZES)*4} stats + {len(WIN_SIZES)} range')


Feature pipeline ready.
  Backward: 14 lags + 5 ROC + 1 accel
  Forward : 23 CGM steps + 23 velocity = 46 cols
  Rolling : 20 stats + 5 range


In [4]:
# ===========================================================================
# CELL 4: Extract Training Shards + Feature Schema + Patient Stats
# ===========================================================================
if SHARD_DIR.exists():
    shutil.rmtree(SHARD_DIR)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

print(f'Streaming train.parquet -> shards (stride={TRAIN_STRIDE}, batch={PATIENT_BATCH})...')
t0 = time.time()

all_pts = pl.scan_parquet(TRAIN_PARQUET).select('id').unique().collect()['id'].to_list()
chunks  = [all_pts[i:i+PATIENT_BATCH] for i in range(0, len(all_pts), PATIENT_BATCH)]
print(f'  {len(all_pts):,} patients | {len(chunks)} batches')

shard_paths = []
total_rows  = 0
for idx, batch in enumerate(chunks):
    lazy  = pl.scan_parquet(TRAIN_PARQUET).filter(pl.col('id').is_in(batch))
    pdf   = _featurise_lazy(lazy, is_train=True, stride=TRAIN_STRIDE)
    pdf   = _add_iob_cob(pdf)
    shard = SHARD_DIR / f'shard_{idx:04d}.parquet'
    pdf.to_parquet(shard, index=False)
    shard_paths.append(shard)
    total_rows += len(pdf)
    del pdf, lazy; gc.collect()
    if (idx+1)%max(1,len(chunks)//5)==0 or idx+1==len(chunks):
        print(f'  Batch {idx+1:3d}/{len(chunks)} | {total_rows:,} rows | [{time.time()-t0:.0f}s]')

print(f'Done: {len(shard_paths)} shards | {total_rows:,} rows')

# Feature schema
EXCLUDE = {
    'id','date','source_file','CGM','CGM_clean','gender','meal_label','workout_label',
    'cgm_device','misc_notes','insulin_delivery_algorithm','insulin_delivery_device',
    'insulin_delivery_modality','insulin_type_basal','insulin_type_bolus','ethnicity',
    'treatment_group','randomization_date','extension_date','is_test','is_pregnant',
    'subject_split_across_traintest','target_30','target_60','target_90','target_120',
}
_sample_cols  = list(pd.read_parquet(shard_paths[0]).columns)
SHARD_SCHEMA  = set(_sample_cols)
BASE_FEAT_COLS = sorted([c for c in _sample_cols if c not in EXCLUDE])
PAT_FEAT_COLS  = ['pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
ALL_FEAT_COLS  = BASE_FEAT_COLS + PAT_FEAT_COLS

# Horizon-specific feature sets — no leakage
def features_for_h(h):
    step_limit = h // CGM_INTERVAL
    bad = set()
    for s in range(step_limit, FL_MAX_STEP+2):
        bad.add(f'cgm_future_{s}')
        bad.add(f'cgm_future_vel_{s}')
    return [c for c in ALL_FEAT_COLS if c not in bad]

FEAT_H = {h: features_for_h(h) for h in HORIZONS}
print(f'\nFeatures: {len(BASE_FEAT_COLS)} base + 4 patient stats = {len(ALL_FEAT_COLS)} total')
for h in HORIZONS:
    fl = len([c for c in FEAT_H[h] if 'future' in c and 'vel' not in c])
    print(f'  h={h:3d}min: {len(FEAT_H[h]):3d} features | FL steps: {fl} (T+5..T+{fl*5})')

# Patient stats
print('\nComputing patient statistics from training shards...')
ps_list = []
for sp in shard_paths:
    tmp = pd.read_parquet(sp, columns=['id','CGM_clean'])
    ps  = tmp.groupby('id')['CGM_clean'].agg(['mean','std','min','max']).reset_index()
    ps.columns = ['id','pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
    ps_list.append(ps); del tmp
pat_stats = pd.concat(ps_list).groupby('id').mean().reset_index()
pat_stats['pat_cgm_std'] = pat_stats['pat_cgm_std'].fillna(30.0)
del ps_list; gc.collect()
print(f'  {len(pat_stats)} training patients')

# Train/val split
unique_pts = np.array(sorted(pat_stats['id'].astype(str).values))
n_val      = max(1, int(len(unique_pts)*0.10))
val_set    = set(unique_pts[-n_val:])
print(f'  Val: {n_val} | Train: {len(unique_pts)-n_val}')


Streaming train.parquet -> shards (stride=12, batch=15)...
  1,183 patients | 79 batches
  Batch  15/79 | 1,733,990 rows | [50s]
  Batch  30/79 | 3,389,189 rows | [91s]
  Batch  45/79 | 5,144,797 rows | [133s]
  Batch  60/79 | 6,985,023 rows | [175s]
  Batch  75/79 | 8,864,125 rows | [218s]
  Batch  79/79 | 9,275,507 rows | [228s]
Done: 79 shards | 9,275,507 rows

Features: 131 base + 4 patient stats = 135 total
  h= 30min:  99 features | FL steps: 5 (T+5..T+25)
  h= 60min: 111 features | FL steps: 11 (T+5..T+55)
  h= 90min: 123 features | FL steps: 17 (T+5..T+85)
  h=120min: 135 features | FL steps: 23 (T+5..T+115)

Computing patient statistics from training shards...
  1183 training patients
  Val: 118 | Train: 1065


In [5]:
# ===========================================================================
# CELL 5: Memory-Safe Training — shard-by-shard per horizon
# Peak RAM: ~5.83 GB (h=120, 135 feats x 9.3M rows) — safe on 16 GB Kaggle
# ===========================================================================
lgbm_models = {}

for h in HORIZONS:
    t_h = time.time()
    print(f'\n--- Training h={h}min ---')
    h_cols  = FEAT_H[h]
    tgt_col = f'target_{h}'
    n_h     = len(h_cols)

    shard_h_cols = [c for c in h_cols if c in SHARD_SCHEMA and c not in PAT_FEAT_COLS]

    print(f'  Pre-allocating {total_rows:,} x {n_h} = {total_rows*n_h*4/1024**3:.2f} GB...')
    X_h   = np.zeros((total_rows, n_h), dtype=np.float32)
    y_h   = np.zeros(total_rows,        dtype=np.float32)
    pid_h = np.empty(total_rows,        dtype=object)
    ptr   = 0

    for sp in shard_paths:
        load_cols_sp = ['id'] + [c for c in shard_h_cols if c in SHARD_SCHEMA]
        if tgt_col in SHARD_SCHEMA:
            load_cols_sp.append(tgt_col)
        df = pd.read_parquet(sp, columns=load_cols_sp)
        df = df.merge(pat_stats, on='id', how='left')
        for c in PAT_FEAT_COLS:
            fill_val = float(df[c].median()) if df[c].notna().any() else 120.0
            df[c] = df[c].fillna(fill_val).astype(np.float32)
        for c in h_cols:
            if c not in df.columns: df[c] = np.float32(0.0)
        n = len(df)
        X_h[ptr:ptr+n]   = df[h_cols].fillna(0.0).to_numpy(np.float32)
        y_h[ptr:ptr+n]   = df[tgt_col].to_numpy(np.float32) if tgt_col in df.columns else np.nan
        pid_h[ptr:ptr+n] = df['id'].astype(str).values
        ptr += n
        del df; gc.collect()

    print(f'  Loaded: {ptr:,} rows [{time.time()-t_h:.0f}s]')

    is_val  = np.array([str(p) in val_set for p in pid_h])
    is_tr   = ~is_val
    tr_mask = is_tr  & ~np.isnan(y_h) & (y_h > 0)
    vl_mask = is_val & ~np.isnan(y_h) & (y_h > 0)
    print(f'  Train: {tr_mask.sum():,} | Val: {vl_mask.sum():,}')

    ds_tr = lgb.Dataset(X_h[tr_mask], label=y_h[tr_mask], feature_name=h_cols, free_raw_data=True)
    ds_vl = lgb.Dataset(X_h[vl_mask], label=y_h[vl_mask], reference=ds_tr,     free_raw_data=True)

    model = lgb.train(
        LGBM_PARAMS, ds_tr, num_boost_round=1000,
        valid_sets=[ds_vl],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    model.save_model(str(OUTPUT_DIR / f'lgbm_h{h}.lgb'))
    lgbm_models[h] = model

    yp = model.predict(X_h[vl_mask]).astype(np.float32)
    fl_n = len([c for c in h_cols if 'future' in c and 'vel' not in c])
    print(f'  h={h:3d}min | {model.num_trees():4d} trees | '
          f'MARD={mard(yp, y_h[vl_mask]):.2f}% | RMSE={rmse(yp, y_h[vl_mask]):.2f} | '
          f'FL steps={fl_n} | [{time.time()-t_h:.0f}s]')

    del X_h, y_h, pid_h, ds_tr, ds_vl, yp; gc.collect()

# Save feature mapping for recovery
feat_h_save = {str(h): cols for h, cols in FEAT_H.items()}
with open(OUTPUT_DIR/'feat_h_cols.json', 'w') as f:
    _json.dump(feat_h_save, f)
print('\nAll models trained and saved. feat_h_cols.json saved.')
shutil.rmtree(SHARD_DIR)



--- Training h=30min ---
  Pre-allocating 9,275,507 x 99 = 3.42 GB...
  Loaded: 9,275,507 rows [25s]
  Train: 9,027,551 | Val: 247,490
[200]	valid_0's l1: 3.67175
[400]	valid_0's l1: 3.61689
[600]	valid_0's l1: 3.59641
[800]	valid_0's l1: 3.58518
[1000]	valid_0's l1: 3.57789
  h= 30min | 1000 trees | MARD=2.44% | RMSE=5.95 | FL steps=5 | [1454s]

--- Training h=60min ---
  Pre-allocating 9,275,507 x 111 = 3.84 GB...
  Loaded: 9,275,507 rows [23s]
  Train: 9,027,106 | Val: 247,444
[200]	valid_0's l1: 3.72473
[400]	valid_0's l1: 3.66306
[600]	valid_0's l1: 3.63818
[800]	valid_0's l1: 3.6259
[1000]	valid_0's l1: 3.61738
  h= 60min | 1000 trees | MARD=2.47% | RMSE=6.16 | FL steps=11 | [1574s]

--- Training h=90min ---
  Pre-allocating 9,275,507 x 123 = 4.25 GB...
  Loaded: 9,275,507 rows [25s]
  Train: 9,026,653 | Val: 247,409
[200]	valid_0's l1: 3.70339
[400]	valid_0's l1: 3.64118
[600]	valid_0's l1: 3.61168
[800]	valid_0's l1: 3.59781
[1000]	valid_0's l1: 3.59004
  h= 90min | 1000 trees

In [6]:
# ===========================================================================
# CELL 6: Predict on Annual Competition Holdout → submission_annual_mard.parquet
#
# Annual template: only 2,228 rows (89 patients) — very fast prediction.
# Batch size: 15 patients at a time for safety.
# ===========================================================================
print('Building features from annual holdout...')
t0 = time.time()

holdout_ids   = pl.scan_parquet(HOLDOUT_PARQUET).select(
    pl.col('id').cast(pl.String)
).unique().collect()['id'].to_list()
h_chunks      = [holdout_ids[i:i+PATIENT_BATCH] for i in range(0, len(holdout_ids), PATIENT_BATCH)]
print(f'  {len(holdout_ids)} holdout patients | {len(h_chunks)} batches')

# Build key lookup from annual template
annual_template['_key'] = (
    annual_template['id'].astype(str)+'||'+annual_template['date'].astype(str)
)
key_to_idx = {k: i for i, k in enumerate(annual_template['_key'])}

# Submission skeleton
submission = annual_template.copy().reset_index(drop=True)
submission.drop(columns=['_key'], inplace=True, errors='ignore')
for h in HORIZONS:
    submission[f'pred_{h}'] = np.nan

# Pat stats from training (already computed in Cell 4)
# For holdout patients not in training, use overall mean
global_cgm_stats = {
    'pat_cgm_mean': float(pat_stats['pat_cgm_mean'].mean()),
    'pat_cgm_std' : float(pat_stats['pat_cgm_std'].mean()),
    'pat_cgm_min' : float(pat_stats['pat_cgm_min'].mean()),
    'pat_cgm_max' : float(pat_stats['pat_cgm_max'].mean()),
}
# Also compute from holdout data itself for better accuracy
print('  Computing holdout patient statistics...')
ps_list2 = []
for batch in h_chunks:
    tmp = pl.scan_parquet(HOLDOUT_PARQUET).filter(
        pl.col('id').cast(pl.String).is_in(batch)
    ).select(['id','CGM']).filter(pl.col('CGM')>0).collect().to_pandas()
    if len(tmp) == 0: continue
    tmp['id'] = tmp['id'].astype(str)
    ps = tmp.groupby('id')['CGM'].agg(['mean','std','min','max']).reset_index()
    ps.columns = ['id','pat_cgm_mean','pat_cgm_std','pat_cgm_min','pat_cgm_max']
    ps_list2.append(ps); del tmp
holdout_pat_stats = pd.concat(ps_list2).groupby('id').mean().reset_index() if ps_list2 else pd.DataFrame()
holdout_pat_stats['pat_cgm_std'] = holdout_pat_stats['pat_cgm_std'].fillna(30.0)
del ps_list2; gc.collect()
print(f'  {len(holdout_pat_stats)} holdout patients with stats')

total_filled = 0
for idx, batch in enumerate(h_chunks):
    lazy = pl.scan_parquet(HOLDOUT_PARQUET).filter(
        pl.col('id').cast(pl.String).is_in(batch)
    )
    pdf = _featurise_lazy(lazy, is_train=False, stride=1)
    pdf = _add_iob_cob(pdf)
    del lazy; gc.collect()

    pdf['id'] = pdf['id'].astype(str)

    # Merge holdout stats (for patients not in training data)
    if len(holdout_pat_stats) > 0:
        pdf = pdf.merge(holdout_pat_stats, on='id', how='left')
    for c in PAT_FEAT_COLS:
        if c not in pdf.columns:
            pdf[c] = np.float32(0.0)
        fill_val = float(pdf[c].median()) if pdf[c].notna().any() else global_cgm_stats.get(c, 120.0)
        pdf[c] = pdf[c].fillna(fill_val).astype(np.float32)

    # Ensure all feature cols exist
    all_h_cols = sorted(set(c for cols in FEAT_H.values() for c in cols))
    for c in all_h_cols:
        if c not in pdf.columns: pdf[c] = np.float32(0.0)

    pdf['_key'] = pdf['id'].astype(str)+'||'+pdf['date'].astype(str)
    pdf = pdf.drop_duplicates(subset='_key', keep='last').reset_index(drop=True)
    pdf['_tmpl_idx'] = pdf['_key'].map(key_to_idx)
    mpdf = pdf.dropna(subset=['_tmpl_idx']).copy()
    mpdf['_tmpl_idx'] = mpdf['_tmpl_idx'].astype(int)

    if len(mpdf) == 0:
        del pdf, mpdf; gc.collect(); continue

    cgm_anc = mpdf['CGM_clean'].fillna(120.0).values.astype(np.float64)
    idxs    = mpdf['_tmpl_idx'].values

    for h in HORIZONS:
        h_cols = FEAT_H[h]
        for c in h_cols:
            if c not in mpdf.columns: mpdf[c] = np.float32(0.0)
        X_b  = mpdf[h_cols].fillna(0.0).to_numpy(np.float32)
        raw  = lgbm_models[h].predict(X_b).astype(np.float64)
        raw  = np.where(np.isnan(raw), cgm_anc, raw)
        mx   = MAX_ROC_PER_MIN * h
        lo   = np.clip(cgm_anc-mx, GLUCOSE_MIN, GLUCOSE_MAX)
        hi   = np.clip(cgm_anc+mx, GLUCOSE_MIN, GLUCOSE_MAX)
        submission.loc[idxs, f'pred_{h}'] = np.clip(raw, lo, hi)

    total_filled += len(mpdf)
    del pdf, mpdf, X_b; gc.collect()
    print(f'  Batch {idx+1}/{len(h_chunks)} | {total_filled} rows filled | [{time.time()-t0:.0f}s]')

# Fill any unmatched rows
for h in HORIZONS:
    ct = submission[f'pred_{h}'].isnull().sum()
    if ct:
        submission.loc[submission[f'pred_{h}'].isnull(), f'pred_{h}'] = 120.0
        print(f'  NOTE: {ct} rows filled with 120.0 for pred_{h}')

# ── Validate ───────────────────────────────────────────────────────────────
submission = submission[['id','source_file','date','pred_30','pred_60','pred_90','pred_120']]
tmpl_clean = annual_template.drop(columns=['_key'], errors='ignore')
null_ct = submission[['pred_30','pred_60','pred_90','pred_120']].isnull().sum().sum()

assert len(submission) == EXPECTED_ROWS,                                             f'Row count {len(submission)} != {EXPECTED_ROWS}'
assert null_ct == 0,                                                                 f'{null_ct} NaN predictions!'
assert (submission['id'].values          == tmpl_clean['id'].values).all(),          'ID mismatch!'
assert (submission['date'].values        == tmpl_clean['date'].values).all(),        'Date mismatch!'
assert (submission['source_file'].values == tmpl_clean['source_file'].values).all(), 'SF mismatch!'

OUT = OUTPUT_DIR / 'submission_annual_mard.parquet'
submission.to_parquet(OUT, index=False)

print(f'\nSAVED: {OUT.name}')
print(f'  Rows   : {len(submission)} (expected {EXPECTED_ROWS})')
print(f'  NaN    : {null_ct}')
for col in ['pred_30','pred_60','pred_90','pred_120']:
    print(f'  {col}: [{submission[col].min():.1f}, {submission[col].max():.1f}]')
display(submission.head(5))


Building features from annual holdout...
  90 holdout patients | 6 batches
  Computing holdout patient statistics...
  90 holdout patients with stats
  Batch 1/6 | 386 rows filled | [4s]
  Batch 2/6 | 683 rows filled | [6s]
  Batch 3/6 | 1088 rows filled | [9s]
  Batch 4/6 | 1520 rows filled | [12s]
  Batch 5/6 | 1848 rows filled | [16s]
  Batch 6/6 | 2228 rows filled | [18s]

SAVED: submission_annual_mard.parquet
  Rows   : 2228 (expected 2228)
  NaN    : 0
  pred_30: [39.1, 401.0]
  pred_60: [39.0, 401.0]
  pred_90: [39.1, 401.0]
  pred_120: [39.1, 401.0]


,id,source_file,date,pred_30,pred_60,pred_90,pred_120
0,annual_competition_secret_holdout_set-1,annual_competition_secret_holdout_set,2021-03-25 03:30:00,183.398978,183.951579,184.159468,184.039749
1,annual_competition_secret_holdout_set-1,annual_competition_secret_holdout_set,2021-03-26 10:10:00,163.168330,163.053066,163.267618,163.111374
2,annual_competition_secret_holdout_set-1,annual_competition_secret_holdout_set,2021-03-27 17:35:00,200.689069,200.277216,200.000808,200.089899
3,annual_competition_secret_holdout_set-1,annual_competition_secret_holdout_set,2021-04-01 00:55:00,170.451708,169.999211,170.147668,170.043389
4,annual_competition_secret_holdout_set-1,annual_competition_secret_holdout_set,2021-04-04 22:35:00,206.342119,206.239658,206.190756,206.012106


In [7]:
# ===========================================================================
# CELL 7: Local Validation using run.py (toolkit scorer)
#
# Annual competition scores are HIDDEN (no public leaderboard score shown).
# This cell runs the toolkit's run.py validator for a local format check.
# ===========================================================================
import subprocess, sys

run_py = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/run.py')
if not run_py.exists():
    for f in Path('/kaggle/input').rglob('run.py'):
        run_py = f; break

OUT_FILE = OUTPUT_DIR / 'submission_annual_mard.parquet'

if run_py.exists():
    print('Running toolkit validator (run.py)...')
    result = subprocess.run(
        [sys.executable, str(run_py),
         '--submission', str(OUT_FILE),
         '--template',   str(ANNUAL_TEMPLATE)],
        capture_output=True, text=True
    )
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:', result.stderr[:500])
else:
    print('run.py not found — doing manual validation only.')

# Manual checks
print('\n--- Submission Summary ---')
sub = pd.read_parquet(OUT_FILE)
print(f'  File   : {OUT_FILE.name}')
print(f'  Size   : {OUT_FILE.stat().st_size / (1024**2):.2f} MB')
print(f'  Rows   : {len(sub):,}  (expected {EXPECTED_ROWS})')
print(f'  Cols   : {list(sub.columns)}')
null_check = sub[['pred_30','pred_60','pred_90','pred_120']].isnull().sum().sum()
print(f'  NaN    : {null_check}  (expected 0)')
for col in ['pred_30','pred_60','pred_90','pred_120']:
    print(f'  {col}: mean={sub[col].mean():.1f} | range=[{sub[col].min():.1f}, {sub[col].max():.1f}]')

print('\n--- Training Validation Scores (NOT annual holdout) ---')
print('  (Annual holdout scores are hidden during competition)')
print()
print('  Val MARD per horizon (from Cell 5 training output):')
print('  Run Cell 5 output to see training validation metrics.')

print('\n--- Output Files ---')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<45s} {f.stat().st_size/(1024**2):.2f} MB')

print('\n--- Next Step ---')
print('  Download: submission_annual_mard.parquet')
print('  Submit at: https://metabonetglucose-leaderboard.hf.space/?tab=annual-leaderboard-tab&view=submit')


run.py not found — doing manual validation only.

--- Submission Summary ---
  File   : submission_annual_mard.parquet
  Size   : 0.11 MB
  Rows   : 2,228  (expected 2228)
  Cols   : ['id', 'source_file', 'date', 'pred_30', 'pred_60', 'pred_90', 'pred_120']
  NaN    : 0  (expected 0)
  pred_30: mean=158.4 | range=[39.1, 401.0]
  pred_60: mean=158.4 | range=[39.0, 401.0]
  pred_90: mean=158.4 | range=[39.1, 401.0]
  pred_120: mean=158.4 | range=[39.1, 401.0]

--- Training Validation Scores (NOT annual holdout) ---
  (Annual holdout scores are hidden during competition)

  Val MARD per horizon (from Cell 5 training output):
  Run Cell 5 output to see training validation metrics.

--- Output Files ---
  __notebook__.ipynb                            0.05 MB
  feat_h_cols.json                              0.01 MB
  lgbm_h120.lgb                                 11.72 MB
  lgbm_h30.lgb                                  11.59 MB
  lgbm_h60.lgb                                  11.64 MB
  lgbm_h9